# LangChain Memory Module
Explores the different types of conversational memory available in LangChain.

In [3]:
!pip install langchain-classic

In [2]:
!pip show langchain
!pip show langchain-core

Name: langchain
Version: 1.2.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/.venv/lib/python3.10/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-core
Version: 1.2.22
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/.venv/lib/python3.10/site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt


In [4]:
from langchain_openai import OpenAI
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationBufferWindowMemory,
    ConversationTokenBufferMemory
)
import tiktoken
from dotenv import load_dotenv
import os

load_dotenv()

llm = OpenAI(
    temperature=0,
    model_name='gpt-3.5-turbo-instruct'
)

##  ConversationBufferMemory
Stores the **full raw conversation history**. Best for short conversations where complete context matters.

| ✅ Pros | ❌ Cons |
|---|---|
| Complete history | High memory usage |
| Accurate references | Slower with long chats |
| Deep context | Not scalable |

In [7]:
# Initialize with full buffer memory
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

# Inspect prompt template
print(conversation.prompt.template)

# Chat
conversation("Good morning AI!")
conversation("My name is Pitsiakos!")
conversation.predict(input="I stay in Newcastle, UK")

# View stored buffer
print(conversation.memory.buffer)

# Test recall
conversation.predict(input="What is my name?")

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Good morning AI!
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Good morning AI!

' Your name is Pitsiakos, as you mentioned earlier. Is there anything else you would like to know?'

## ConversationBufferWindowMemory
Stores only the **last K interactions**. Older ones are dropped to save memory.

| ✅ Pros | ❌ Cons |
|---|---|
| Efficient memory | Loses old context |
| Low token count | Shallower understanding |
| Up-to-date context | Limited history |

In [10]:
# k=3 — only last 3 exchanges are kept
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferWindowMemory(k=3)
)

# Inspect prompt template
print(conversation.prompt.template)

# Chat
conversation.predict(input="Good morning AI!")
conversation.predict(input="My Name is Pitsiakkos")
conversation.predict(input="I stay in Newcastle, UK")

# View current window
print(conversation.memory.buffer)

# Test recall
conversation.predict(input="What is my name?")

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Good morning AI!
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Good morning AI!

' Your name is Pitsiakkos, as you mentioned earlier. Is there anything else you would like to know about yourself?'